<a href="https://colab.research.google.com/github/Nouha50215/NLP-Assignment-3/blob/main/darija_ngram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Darija N-gram Language Model
Trains a probabilistic n-gram model on Moroccan Darija.

Mounting Google Drive

In [1]:
from google.colab import drive
# Mount Google Drive so we can access files stored in it
drive.mount('/content/drive')
print('✅ Drive mounted at /content/drive')

Mounted at /content/drive
✅ Drive mounted at /content/drive


Setting the data folder path

In [2]:
import os

DATA_DIR = '/content/drive/MyDrive/data'

# Walking through the folder and list all files found
if os.path.isdir(DATA_DIR):
    files = [os.path.join(r,f) for r,_,fs in os.walk(DATA_DIR) for f in fs]
    print(f'✅ Found {len(files)} file(s) in {DATA_DIR}')
    for f in files[:20]: print(' ', f)
    if len(files) > 20: print(f'  ... and {len(files)-20} more')
else:
    print(f'❌ Folder not found: {DATA_DIR}')

✅ Found 43 file(s) in /content/drive/MyDrive/data
  /content/drive/MyDrive/data/twitter/tweets2.txt
  /content/drive/MyDrive/data/twitter/tweets3.txt
  /content/drive/MyDrive/data/twitter/tweets4.txt
  /content/drive/MyDrive/data/twitter/tweets.txt
  /content/drive/MyDrive/data/Youtube/amlignore
  /content/drive/MyDrive/data/Youtube/DS_Store
  /content/drive/MyDrive/data/Youtube/YouTube Data 1.txt
  /content/drive/MyDrive/data/goud.ma/goud_articles_1.txt
  /content/drive/MyDrive/data/goud.ma/goud_articles_2.txt
  /content/drive/MyDrive/data/goud.ma/goud_articles_3.txt
  /content/drive/MyDrive/data/goud.ma/goud_articles_4.txt
  /content/drive/MyDrive/data/goud.ma/goud_articles_5.txt
  /content/drive/MyDrive/data/goud.ma/goud_articles_6.txt
  /content/drive/MyDrive/data/goud.ma/goud_articles_7.txt
  /content/drive/MyDrive/data/goud.ma/goud_articles_8.txt
  /content/drive/MyDrive/data/goud.ma/links_processed.txt
  /content/drive/MyDrive/data/goud.ma/goud_articles_9.txt
  /content/drive/My

Settings

In [3]:
# N-gram order: how many words the model looks back at — 3 is a good default
N = 3

# Smoothing method: kneser_ney is recommended, laplace is simpler
SMOOTHING = 'kneser_ney'

# Fraction of data used for training, dev, and test
TRAIN_RATIO = 0.80
DEV_RATIO   = 0.10

# Max lines to read — keeps RAM usage safe on free Colab (stays well under 12.7 GB)
MAX_LINES = 500_000

# Number of sentences to sample when computing perplexity — 500 is fast and representative
EVAL_SAMPLE = 500

# Field name to extract text from if your files are JSON or JSONL
JSON_TEXT_FIELD = 'text'

print(f'Settings: n={N}, smoothing={SMOOTHING}, max_lines={MAX_LINES:,}, eval_sample={EVAL_SAMPLE}')

Settings: n=3, smoothing=kneser_ney, max_lines=500,000, eval_sample=500


Defining functions and model

In [4]:
import re, math, random, json
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Tuple, Optional

# Tokenization

def normalize_darija(text: str) -> str:
    # Unify different Arabic letter forms into one standard form
    text = text.replace('أ','ا').replace('إ','ا').replace('آ','ا')
    # Normalize ta marbuta and alef maqsura
    text = text.replace('ة','ه').replace('ى','ي')
    # Remove zero-width characters that cause tokenization issues
    text = text.replace('\u200c',' ').replace('\u200d','')
    # Strip Arabic diacritics (short vowel marks)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    # Collapse multiple spaces into one
    return re.sub(r'\s+', ' ', text).strip()

def tokenize(text: str) -> List[str]:
    # Extract Arabic-script tokens and Latin/Arabizi tokens separately
    tokens = re.findall(r'[\u0600-\u06FF\u0750-\u077F]+|[a-zA-Z0-9\']+',
                        normalize_darija(text))
    # Return lowercased tokens, skipping empty strings
    return [t.lower() for t in tokens if t]

def add_boundaries(tokens: List[str], n: int) -> List[str]:
    # Wrap tokens with start markers and one end marker for n-gram counting
    return ['<s>'] * (n-1) + tokens + ['</s>']

# Data loading

def read_file(path: str) -> str:
    # Try multiple encodings to handle Arabic and mixed-script files
    for enc in ('utf-8', 'utf-8-sig', 'latin-1', 'cp1256'):
        try:
            with open(path, encoding=enc) as f: return f.read()
        except UnicodeDecodeError: pass
    return ''

def parse_content(content: str, filename: str) -> str:
    # Extract plain text from JSON or JSONL files using the configured field name
    ext = Path(filename).suffix.lower()
    if ext == '.json':
        try:
            data = json.loads(content)
            if isinstance(data, list):
                return ' '.join(str(r.get(JSON_TEXT_FIELD,'')) for r in data if isinstance(r,dict))
            return str(data.get(JSON_TEXT_FIELD, content))
        except: pass
    elif ext == '.jsonl':
        lines = []
        for line in content.splitlines():
            try: lines.append(str(json.loads(line).get(JSON_TEXT_FIELD,'')))
            except: pass
        return ' '.join(lines)
    # For .txt and .csv files return content as-is
    return content

def load_corpus(data_dir: str, max_lines: int = 500_000) -> List[List[str]]:
    supported = {'.txt', '.csv', '.json', '.jsonl'}
    all_sentences, total_tokens = [], 0
    file_count, line_count = 0, 0

    for root, dirs, files in os.walk(data_dir):
        # Stop walking folders if we already hit the line limit
        if line_count >= max_lines: break
        for fname in sorted(files):
            # Stop reading files if we already hit the line limit
            if line_count >= max_lines:
                print(f'  ⚠ Reached MAX_LINES={max_lines:,} — stopping early to save RAM.')
                break
            # Skip unsupported file types
            if not any(fname.lower().endswith(e) for e in supported): continue
            fpath = os.path.join(root, fname)
            content = read_file(fpath)
            text = parse_content(content, fname)
            for line in text.splitlines():
                # Stop adding lines once we hit the cap
                if line_count >= max_lines: break
                toks = tokenize(line)
                if toks:
                    all_sentences.append(toks)
                    total_tokens += len(toks)
                    line_count   += 1
            file_count += 1
            print(f'    [OK] {fname}')

    print(f'  Loaded {file_count} file(s) → {len(all_sentences):,} sentences | {total_tokens:,} tokens')
    return all_sentences

# Train/dev/test split

def split_data(sentences, train_ratio=0.80, dev_ratio=0.10, seed=42):
    # Shuffle with a fixed seed so results are reproducible
    random.seed(seed)
    data = sentences[:]
    random.shuffle(data)
    n = len(data)
    t = int(n * train_ratio)
    d = int(n * (train_ratio + dev_ratio))
    # Return three non-overlapping slices
    return data[:t], data[t:d], data[d:]

# N-gram model

class NgramLM:
    """Probabilistic n-gram LM with Laplace or Kneser-Ney smoothing."""

    def __init__(self, n=3, smoothing='kneser_ney', laplace_k=1.0, kn_discount=0.75):
        self.n, self.smoothing       = n, smoothing
        self.laplace_k, self.kn_discount = laplace_k, kn_discount
        # Stores count of each n-gram seen during training
        self.ngram_counts            = Counter()
        # Stores count of each (n-1)-gram context seen during training
        self.context_counts          = Counter()
        # Stores count of each individual word
        self.unigram_counts          = Counter()
        # Stores how many unique contexts each word appears in (for Kneser-Ney)
        self.continuation_counts     = Counter()
        # Stores precomputed unique follower counts per context (makes KN fast)
        self.unique_follower_counts  = {}
        self.vocab                   = set()
        self.total_tokens            = 0
        self.total_bigram_types      = 0

    def train(self, sentences):
        print(f'Training {self.n}-gram model ({self.smoothing})...')
        bigram_set = set()
        # Temporary dict to collect unique followers per context efficiently
        follower_sets = defaultdict(set)

        for tokens in sentences:
            # Add start/end markers around each sentence
            padded = add_boundaries(tokens, self.n)
            self.vocab.update(tokens)
            for i, w in enumerate(padded):
                # Count every word seen
                self.unigram_counts[w] += 1
                self.total_tokens      += 1
                if i >= self.n-1:
                    ng  = tuple(padded[i-self.n+1:i+1])
                    ctx = ng[:-1]
                    # Count this n-gram and its context
                    self.ngram_counts[ng]    += 1
                    self.context_counts[ctx] += 1
                    # Record that word w follows context ctx
                    follower_sets[ctx].add(w)
                if self.smoothing == 'kneser_ney' and i >= 1:
                    bigram_set.add((padded[i-1], w))

        if self.smoothing == 'kneser_ney':
            # Count how many unique left contexts each word appears in
            for (_, w) in bigram_set:
                self.continuation_counts[w] += 1
            self.total_bigram_types = len(bigram_set)

        # Precompute unique follower counts once — avoids expensive per-call scanning
        self.unique_follower_counts = {ctx: len(words) for ctx, words in follower_sets.items()}

        # Add special tokens to vocabulary
        self.vocab.update(['<s>', '</s>', '<UNK>'])
        print(f'  Vocab: {len(self.vocab):,} types | Tokens: {self.total_tokens:,}')

    def _prob(self, ngram, context):
        # Route to the correct smoothing method
        return self._laplace(ngram, context) if self.smoothing == 'laplace' \
               else self._kneser_ney(ngram, context, self.n)

    def _laplace(self, ngram, context):
        # Add-k smoothing: add laplace_k to every count to avoid zero probabilities
        V     = len(self.vocab)
        c_ctx = self.context_counts.get(context, 0) if context else self.total_tokens
        return (self.ngram_counts.get(ngram, 0) + self.laplace_k) / (c_ctx + self.laplace_k * V)

    def _kneser_ney(self, ngram, context, order):
        # Interpolated Kneser-Ney: discounts high counts and redistributes mass to lower orders
        d, V = self.kn_discount, self.total_bigram_types or 1
        if order == 1:
            # Base case: use continuation probability (how many contexts this word appears in)
            return self.continuation_counts.get(ngram[-1], 0) / V
        c_ng  = self.ngram_counts.get(ngram, 0)
        c_ctx = self.context_counts.get(context, 0) if context else self.total_tokens
        # Use precomputed follower counts — this is what makes it fast
        uniq  = self.unique_follower_counts.get(context, 0)
        lam   = (d * uniq / c_ctx) if c_ctx else 1.0
        # Recursively back off to lower order
        return max(c_ng - d, 0) / (c_ctx or 1) + lam * self._kneser_ney(ngram[1:], context[1:], order-1)

    def log_prob_sentence(self, tokens):
        # Compute the log base-2 probability of a full sentence
        padded = add_boundaries(tokens, self.n)
        logp   = 0.0
        for i in range(self.n-1, len(padded)):
            ng  = tuple(padded[i-self.n+1:i+1])
            p   = self._prob(ng, ng[:-1])
            logp += math.log2(p) if p > 0 else -1e9
        return logp

    def perplexity(self, sentences):
        # Perplexity measures how well the model predicts the data — lower is better
        logp, words = 0.0, 0
        for s in sentences:
            if s: logp += self.log_prob_sentence(s); words += len(s)
        if not words: return float('inf')
        try: return 2 ** (-logp / words)
        except OverflowError: return float('inf')

    def generate(self, max_tokens=40, temperature=1.0, seed_tokens=None):
        # Start with sentence-start markers, optionally followed by seed words
        ctx    = (['<s>'] * (self.n-1) + seed_tokens)[-(self.n-1):] \
                 if seed_tokens else ['<s>'] * (self.n-1)
        result = seed_tokens[:] if seed_tokens else []
        for _ in range(max_tokens):
            c = tuple(ctx[-(self.n-1):]) if self.n > 1 else ()
            # Compute probability of every word in vocabulary given current context
            cands = {w: self._prob(c+(w,), c) for w in self.vocab if w != '<s>'}
            total = sum(cands.values())
            if not total: break
            # Apply temperature to control randomness
            if temperature != 1.0:
                cands = {w: p**(1/temperature) for w,p in cands.items()}
                total = sum(cands.values())
            # Sample the next word using weighted random selection
            r, cum, chosen = random.random() * total, 0.0, '</s>'
            for w, p in cands.items():
                cum += p
                if cum >= r: chosen = w; break
            # Stop if we sampled the end-of-sentence marker
            if chosen == '</s>': break
            result.append(chosen); ctx.append(chosen)
        return ' '.join(result)

    def save(self, path):
        # Serialize the model to a JSON file so it can be reloaded without retraining
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({
                'n': self.n, 'smoothing': self.smoothing,
                'laplace_k': self.laplace_k, 'kn_discount': self.kn_discount,
                'ngram_counts':           {str(k): v for k,v in self.ngram_counts.items()},
                'context_counts':         {str(k): v for k,v in self.context_counts.items()},
                'unigram_counts':         dict(self.unigram_counts),
                'continuation_counts':    dict(self.continuation_counts),
                'unique_follower_counts': {str(k): v for k,v in self.unique_follower_counts.items()},
                'vocab':                  list(self.vocab),
                'total_tokens':           self.total_tokens,
                'total_bigram_types':     self.total_bigram_types,
            }, f, ensure_ascii=False, indent=2)
        print(f'Model saved → {path}')

    @classmethod
    def load(cls, path):
        # Reload a previously saved model from JSON
        with open(path, encoding='utf-8') as f: data = json.load(f)
        m = cls(n=data['n'], smoothing=data['smoothing'],
                laplace_k=data['laplace_k'], kn_discount=data['kn_discount'])
        m.ngram_counts           = Counter({eval(k): v for k,v in data['ngram_counts'].items()})
        m.context_counts         = Counter({eval(k): v for k,v in data['context_counts'].items()})
        m.unigram_counts         = Counter(data['unigram_counts'])
        m.continuation_counts    = Counter(data['continuation_counts'])
        m.unique_follower_counts = {eval(k): v for k,v in data.get('unique_follower_counts',{}).items()}
        m.vocab                  = set(data['vocab'])
        m.total_tokens           = data['total_tokens']
        m.total_bigram_types     = data['total_bigram_types']
        return m

print('✅ All functions and classes defined.')

✅ All functions and classes defined.


 Loading data from Drive

In [5]:
print('Loading corpus from:', DATA_DIR)
print('─' * 50)

# Load and tokenize all files from the Drive folder up to MAX_LINES
all_sentences = load_corpus(DATA_DIR, max_lines=MAX_LINES)

if not all_sentences:
    raise RuntimeError('No sentences loaded. Check DATA_DIR and file types.')

# Split into train, dev, and test sets
train_sents, dev_sents, test_sents = split_data(all_sentences, TRAIN_RATIO, DEV_RATIO)
print(f'Split → train: {len(train_sents):,} | dev: {len(dev_sents):,} | test: {len(test_sents):,}')

# Delete the raw list and run garbage collection to free RAM before training
del all_sentences
import gc; gc.collect()
print('✅ RAM freed after split.')

Loading corpus from: /content/drive/MyDrive/data
──────────────────────────────────────────────────
    [OK] tweets.txt
    [OK] tweets2.txt
  ⚠ Reached MAX_LINES=500,000 — stopping early to save RAM.
  Loaded 2 file(s) → 500,000 sentences | 4,886,040 tokens
Split → train: 400,000 | dev: 50,000 | test: 50,000
✅ RAM freed after split.


Training the model

In [6]:
# Create and train the model using the settings defined in cell
model = NgramLM(n=N, smoothing=SMOOTHING)
model.train(train_sents)

Training 3-gram model (kneser_ney)...
  Vocab: 510,682 types | Tokens: 5,109,766


Evaluating (perplexity PP)

In [7]:
import random
random.seed(42)

# Sample a small subset from each split — fast and still statistically representative
train_sample = random.sample(train_sents, min(EVAL_SAMPLE, len(train_sents)))
dev_sample   = random.sample(dev_sents,   min(EVAL_SAMPLE, len(dev_sents)))
test_sample  = random.sample(test_sents,  min(EVAL_SAMPLE, len(test_sents)))

# Perplexity measures how well the model predicts unseen text — lower is better
print('Perplexity')
print('─' * 40)
print(f'  Train : {model.perplexity(train_sample):,.2f}')
print(f'  Dev   : {model.perplexity(dev_sample):,.2f}')
print(f'  Test  : {model.perplexity(test_sample):,.2f}')

Perplexity
────────────────────────────────────────
  Train : 20.23
  Dev   : inf
  Test  : inf


Inspecting what the model learned

In [8]:
# Print a header for the n-grams section
print(f'Top 20 most frequent {N}-grams:')
# Print a divider line
print('─' * 50)
# Loop through the 20 most common n-grams with their counts
for i, (ng, cnt) in enumerate(model.ngram_counts.most_common(20), 1):
    # Join the n-gram words into a string and print with its count
    print(f'  {i:2}. {" ".join(ng):<35} {cnt:,}')

# Print a header for the words section
print(f'\nTop 30 most frequent words:')
# Print a divider line
print('─' * 50)
# Get the 35 most common words, filter out internal model markers, keep only 30
top = [(w,c) for w,c in model.unigram_counts.most_common(35)
       if w not in ('<s>','</s>','<UNK>')][:30]  # <s>=start, </s>=end, <UNK>=unknown word
# Loop through the filtered words and print each with its count
for i, (w, cnt) in enumerate(top, 1):
    print(f'  {i:2}. {w:<30} {cnt:,}')

Top 20 most frequent 3-grams:
──────────────────────────────────────────────────
   1. https t co                          63,576
   2. <s> <s> شوف                         3,625
   3. <s> <s> انا                         2,728
   4. <s> <s> https                       2,521
   5. <s> https t                         2,521
   6. <s> <s> من                          2,403
   7. <s> <s> i                           2,188
   8. <s> <s> ان                          1,826
   9. ا https t                           1,797
  10. <s> <s> لا                          1,755
  11. <s> <s> اللهم                       1,730
  12. <s> <s> صباح                        1,681
  13. <s> <s> و                           1,666
  14. <s> <s> بغيت                        1,660
  15. <s> <s> يا                          1,563
  16. تطبيق يشجع ع                        1,278
  17. يشجع ع المشي                        1,278
  18. ع المشي عن                          1,278
  19. المشي عن طريق                       1,278
  20. 

Generating sentences

In [12]:
# Optional seed text to start generation from — set to None to start from scratch
SEED_TEXT   = None   # e.g. 'واش كتعرف' or 'wach nta'
# How many sentences to generate
NUM_SAMPLES = 5
# Temperature controls randomness: >1 more random, <1 more focused
TEMPERATURE = 1.0

# Tokenize the seed text if provided
seed_tokens = tokenize(SEED_TEXT) if SEED_TEXT else None

print(f'Generated sentences (n={model.n}, {model.smoothing}):')
print('─' * 50)
for i in range(NUM_SAMPLES):
    # Generate one sentence and print it
    print(f'  [{i+1}]', model.generate(max_tokens=30, temperature=TEMPERATURE,
                                       seed_tokens=seed_tokens))

Generated sentences (n=3, kneser_ney):
──────────────────────────────────────────────────
  [1] سؤال يستحق مع i tesra ليش
  [2] les من كيفي فرحان nothing چیز والالم et
  [3] عسانا نموت ولانشوف المذله شوف
  [4] place au httpstcopbr6ukxfu6 ghadi bosse httpstcojhi5swwa7i طيزك حتا فيهو في floor اللعيبه تمام 9di1qcrljh
  [5] hola متصل


Saving the model to Drive


In [11]:
# Path where the trained model will be saved in your Google Drive
SAVE_PATH = '/content/drive/MyDrive/darija_ngram_model.json'

# Save the model so you can reload it later without retraining
model.save(SAVE_PATH)
print('Done!')

Model saved → /content/drive/MyDrive/darija_ngram_model.json
Done!
